In [1]:
# Module 6: Path to Production
# Lab: Session Resumption — Crash Recovery

# Setup -- install dependencies (run once per session)
# !pip install -q claude-agent-sdk python-dotenv

In [2]:
# Import libraries
import os
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    ResultMessage,
    get_session_messages, list_sessions,
)

In [3]:
# Load API keys from .env file
load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print(f"Anthropic key (SDK): {'Yes' if ANTHROPIC_API_KEY else 'No'}")

Anthropic key (SDK): Yes


In [4]:
# Step 1 -- Run agent with a turn limit to simulate a crash
# max_turns=2 forces early termination; session_id is captured from ResultMessage
async def run_with_crash(task: str):
    """Run agent with a low turn limit to simulate a crash."""
    options = ClaudeAgentOptions(
        allowed_tools=["Read", "Glob", "Grep", "Edit"],
        max_turns=2,
        model="claude-haiku-4-5-20251001",
    )
    session_id = None
    try:
        async for message in query(prompt=task, options=options):
            if isinstance(message, ResultMessage):
                session_id = message.session_id
                if message.subtype == "success":
                    print(f"[Done] {message.result[:200]}")
    except Exception as e:
        print(f"[Crash] {e}")
    return session_id

In [5]:
# Step 2 -- Inspect session history after the crash
# Use get_session_messages() to read what the agent did without running it
def inspect_session(session_id: str):
    """Print the conversation history from a session."""
    messages = get_session_messages(session_id)
    print(f"\n--- Session {session_id[:8]}... ({len(messages)} messages) ---")
    for i, msg in enumerate(messages):
        role = msg.type.upper()
        preview = str(msg.message)[:120]
        print(f"  [{i}] {role}: {preview}")
    return messages

In [6]:
# Step 3 -- Resume the crashed session
# Pass resume=<session_id> in ClaudeAgentOptions to continue from last state
async def resume_session(session_id: str, follow_up: str):
    """Resume a session from its last state."""
    options = ClaudeAgentOptions(
        allowed_tools=["Read", "Glob", "Grep", "Edit"],
        resume=session_id,
        model="claude-haiku-4-5-20251001",
    )
    async for message in query(prompt=follow_up, options=options):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(f"[Resumed] {message.result[:300]}")

In [7]:
# Step 4 -- Run the full crash-and-recover pipeline
# 1. Run with max_turns=2 to trigger early termination
# 2. Inspect the captured session history
# 3. Resume with the same session ID to continue work
# Resolve data paths to absolute so the model can't misinterpret them
DATA_DIR = Path("data").resolve()
TASK = f"Read {DATA_DIR}/task_state.json and {DATA_DIR}/work_in_progress.txt, then continue the refactoring work described."
FOLLOW_UP = "Continue exactly where you left off."

async def main():
    session_id = await run_with_crash(TASK)
    if session_id:
        inspect_session(session_id)
        await resume_session(session_id, FOLLOW_UP)

await main()

[Crash] Claude Code returned an error result: Reached maximum number of turns (2)

--- Session 4ac87068... (9 messages) ---
  [0] USER: {'role': 'user', 'content': 'Read ./data/task_state.json and ./data/work_in_progress.txt, then continue the refactoring 
  [1] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': 'msg_011CdXyKRDbccJsBGm3Mn7bY', 'type': 'message', 'role': 'assistant', 'co
  [2] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': 'msg_011CdXyKRDbccJsBGm3Mn7bY', 'type': 'message', 'role': 'assistant', 'co
  [3] USER: {'role': 'user', 'content': [{'type': 'tool_result', 'content': 'File does not exist. Note: your current working directo
  [4] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': 'msg_011CdXyKRDbccJsBGm3Mn7bY', 'type': 'message', 'role': 'assistant', 'co
  [5] USER: {'role': 'user', 'content': [{'type': 'tool_result', 'content': 'File does not exist. Note: your current working directo
  [6] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': '

CancelledError: 

In [ ]:
# Step 5 -- List all available sessions on disk
# list_sessions() reads from ~/.claude/projects/<encoded-cwd>/
sessions = list_sessions()
print(f"\n--- All Sessions ({len(sessions)}) ---")
for s in sessions:
    print(f"  {s.session_id[:12]}... | {s.first_prompt[:60]} | {s.created_at}")